# BeautifulSoup 网页解析教程

本教程介绍如何使用 BeautifulSoup 库解析 HTML 和提取数据。

## 目录
1. BeautifulSoup 基础
2. 查找元素
3. 提取数据
4. 实战：爬取电影信息
5. 实战：爬取贴吧内容

## 安装依赖

```bash
pip install beautifulsoup4 lxml
```

## 1. BeautifulSoup 基础

In [ ]:
from bs4 import BeautifulSoup
import urllib.request

# 示例 HTML
html_doc = """
<html>
<head><title>示例页面</title></head>
<body>
    <div class="container">
        <h1 id="main-title">欢迎学习爬虫</h1>
        <p class="intro">这是一个示例段落</p>
        <ul class="list">
            <li>项目 1</li>
            <li>项目 2</li>
            <li>项目 3</li>
        </ul>
    </div>
</body>
</html>
"""

# 创建 BeautifulSoup 对象
soup = BeautifulSoup(html_doc, 'lxml')

# 美化输出
print("格式化的 HTML:")
print(soup.prettify()[:300])

## 2. 查找元素的方法

In [ ]:
from bs4 import BeautifulSoup

html = """
<div class="movie">
    <h2 class="title">流浪地球</h2>
    <span class="score">8.5</span>
    <p class="desc">科幻电影</p>
</div>
<div class="movie">
    <h2 class="title">三体</h2>
    <span class="score">9.0</span>
    <p class="desc">科幻小说改编</p>
</div>
"""

soup = BeautifulSoup(html, 'lxml')

# 方法 1: find() - 查找第一个匹配的元素
first_movie = soup.find('div', class_='movie')
print("第一个电影:", first_movie.find('h2').get_text())

# 方法 2: find_all() - 查找所有匹配的元素
all_movies = soup.find_all('div', class_='movie')
print(f"\n找到 {len(all_movies)} 部电影")

# 方法 3: select() - 使用 CSS 选择器
titles = soup.select('.movie .title')
print("\n所有标题:")
for title in titles:
    print(f"  - {title.get_text()}")

# 方法 4: 使用属性字典
movies_with_attrs = soup.find_all('div', attrs={'class': 'movie'})
print(f"\n使用 attrs 找到 {len(movies_with_attrs)} 部电影")

## 3. 提取数据的技巧

In [ ]:
from bs4 import BeautifulSoup

html = """
<div class="item" data-id="123">
    <a href="/movie/123" title="流浪地球">流浪地球</a>
    <span class="rating">评分: <em>8.5</em></span>
    <p class="info">导演: 郭帆 | 主演: 吴京</p>
</div>
"""

soup = BeautifulSoup(html, 'lxml')
item = soup.find('div', class_='item')

# 获取文本内容
title = item.find('a').get_text()
print(f"标题: {title}")

# 获取属性值
link = item.find('a')['href']
data_id = item['data-id']
print(f"链接: {link}")
print(f"数据ID: {data_id}")

# 获取嵌套元素的文本
rating = item.find('span', class_='rating').find('em').get_text()
print(f"评分: {rating}")

# 使用 strip() 清理文本
info = item.find('p', class_='info').get_text().strip()
print(f"信息: {info}")

# 使用 get() 方法安全获取属性
title_attr = item.find('a').get('title', '无标题')
print(f"标题属性: {title_attr}")

## 4. 实战：电影信息爬虫

基于原始代码改编的电影爬虫

In [ ]:
from bs4 import BeautifulSoup
import urllib.request
import re
from datetime import datetime

class MovieItem:
    """电影数据模型"""
    def __init__(self):
        self.name = None
        self.score = None
        self.starring = None
    
    def __repr__(self):
        return f"Movie(name={self.name}, score={self.score}, starring={self.starring})"

class MovieCrawler:
    """电影信息爬虫"""
    
    def __init__(self, base_url):
        self.base_url = base_url
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        self.items = []
    
    def fetch_page(self, url):
        """获取页面内容"""
        request = urllib.request.Request(url, headers=self.headers)
        try:
            response = urllib.request.urlopen(request, timeout=10)
            content = response.read().decode('utf-8')
            print(f"✓ 成功获取: {url}")
            return content
        except Exception as e:
            print(f"✗ 获取失败: {e}")
            return None
    
    def parse_movie_list(self, html_content):
        """解析电影列表"""
        if not html_content:
            return []
        
        soup = BeautifulSoup(html_content, 'lxml')
        movies = []
        
        # 这里需要根据实际网站结构调整选择器
        # 示例：查找所有电影条目
        movie_tags = soup.find_all('div', class_='movie-item')
        
        for tag in movie_tags:
            item = MovieItem()
            
            # 提取电影名称
            name_tag = tag.find('span', class_='title')
            if name_tag:
                item.name = name_tag.get_text().strip()
            
            # 提取评分
            score_tag = tag.find('span', class_='score')
            if score_tag:
                item.score = score_tag.get_text().strip().replace('分', '')
            
            # 提取主演
            starring_tag = tag.find('span', class_='starring')
            if starring_tag:
                item.starring = starring_tag.get_text().strip().replace('主演：', '')
            
            if item.name:  # 只添加有名称的电影
                movies.append(item)
                print(f"  解析: {item.name}")
        
        return movies
    
    def save_to_file(self, filename='movies.txt'):
        """保存到文件"""
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(f"爬取时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write("="*60 + "\n")
            f.write(f"{'电影名称':<20} {'评分':<10} {'主演'}\n")
            f.write("-"*60 + "\n")
            
            for item in self.items:
                f.write(f"{item.name:<20} {item.score:<10} {item.starring}\n")
        
        print(f"\n✓ 已保存 {len(self.items)} 条记录到 {filename}")
    
    def run(self, max_pages=1):
        """运行爬虫"""
        print(f"开始爬取电影信息...")
        print("="*60)
        
        for page in range(1, max_pages + 1):
            url = f"{self.base_url}?page={page}"
            html = self.fetch_page(url)
            movies = self.parse_movie_list(html)
            self.items.extend(movies)
        
        print("="*60)
        print(f"爬取完成！共获取 {len(self.items)} 部电影")
        return self.items

# 使用示例（需要替换为实际的URL）
print("MovieCrawler 类已定义")
print("\n使用方法:")
print("crawler = MovieCrawler('http://example.com/movies')")
print("movies = crawler.run(max_pages=3)")
print("crawler.save_to_file('movies.txt')")

## 5. 实战：贴吧内容爬虫

In [ ]:
from bs4 import BeautifulSoup
import urllib.request
import urllib.parse

class TiebaItem:
    """贴吧帖子数据模型"""
    def __init__(self):
        self.title = None          # 帖子标题
        self.author = None         # 作者
        self.create_time = None    # 创建时间
        self.reply_count = None    # 回复数
        self.content = None        # 内容摘要
        self.last_author = None    # 最后回复者
        self.last_time = None      # 最后回复时间

class TiebaCrawler:
    """百度贴吧爬虫"""
    
    def __init__(self, keyword, max_pages=3):
        self.keyword = keyword
        self.max_pages = max_pages
        self.base_url = 'http://tieba.baidu.com/f'
        self.items = []
    
    def build_url(self, page_num):
        """构建URL"""
        pn = (page_num - 1) * 50
        params = {
            'kw': self.keyword,
            'ie': 'utf-8',
            'pn': pn
        }
        return f"{self.base_url}?{urllib.parse.urlencode(params)}"
    
    def fetch_page(self, url):
        """获取页面"""
        try:
            response = urllib.request.urlopen(url, timeout=10)
            content = response.read().decode('utf-8')
            print(f"✓ 获取页面成功")
            return content
        except Exception as e:
            print(f"✗ 获取失败: {e}")
            return None
    
    def parse_posts(self, html_content):
        """解析帖子列表"""
        if not html_content:
            return []
        
        soup = BeautifulSoup(html_content, 'lxml')
        posts = []
        
        # 查找所有帖子（需要根据实际页面结构调整）
        post_tags = soup.find_all('li', class_='j_thread_list')
        
        for tag in post_tags:
            try:
                item = TiebaItem()
                
                # 标题
                title_tag = tag.find('a', class_='j_th_tit')
                if title_tag:
                    item.title = title_tag.get_text().strip()
                
                # 作者
                author_tag = tag.find('span', class_='frs-author-name-wrap')
                if author_tag and author_tag.find('a'):
                    item.author = author_tag.find('a').get_text().strip()
                
                # 回复数
                reply_tag = tag.find('span', attrs={'title': '回复'})
                if reply_tag:
                    item.reply_count = reply_tag.get_text().strip()
                
                if item.title:
                    posts.append(item)
                    print(f"  解析: {item.title[:30]}...")
            
            except Exception as e:
                print(f"  解析帖子出错: {e}")
                continue
        
        return posts
    
    def save_to_file(self, filename):
        """保存到文件"""
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(f"贴吧: {self.keyword}\n")
            f.write(f"爬取时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write("="*80 + "\n\n")
            
            for item in self.items:
                f.write(f"标题: {item.title}\n")
                f.write(f"作者: {item.author}\n")
                f.write(f"回复数: {item.reply_count}\n")
                f.write("-"*80 + "\n\n")
        
        print(f"\n✓ 已保存到 {filename}")
    
    def run(self):
        """运行爬虫"""
        print(f"开始爬取贴吧: {self.keyword}")
        print("="*60)
        
        for page in range(1, self.max_pages + 1):
            print(f"\n第 {page} 页:")
            url = self.build_url(page)
            html = self.fetch_page(url)
            posts = self.parse_posts(html)
            self.items.extend(posts)
        
        print("="*60)
        print(f"爬取完成！共获取 {len(self.items)} 个帖子")
        return self.items

# 使用示例
print("TiebaCrawler 类已定义")
print("\n使用方法:")
print("crawler = TiebaCrawler('Python', max_pages=3)")
print("posts = crawler.run()")
print("crawler.save_to_file('tieba_posts.txt')")

## 6. 常用技巧总结

In [ ]:
from bs4 import BeautifulSoup
import re

# 技巧集合
html = """
<div class="container">
    <p class="text">  这是一段文本  </p>
    <a href="/page1" data-id="123">链接1</a>
    <a href="/page2" data-id="456">链接2</a>
    <span class="price">¥99.00</span>
</div>
"""

soup = BeautifulSoup(html, 'lxml')

print("BeautifulSoup 常用技巧:")
print("="*50)

# 1. 清理空白字符
text = soup.find('p', class_='text').get_text(strip=True)
print(f"1. 清理空白: '{text}'")

# 2. 获取所有链接
links = [a['href'] for a in soup.find_all('a')]
print(f"2. 所有链接: {links}")

# 3. 使用正则表达式匹配属性
data_items = soup.find_all('a', attrs={'data-id': re.compile(r'\d+')})
print(f"3. 匹配 data-id: {len(data_items)} 个")

# 4. 提取数字
price_text = soup.find('span', class_='price').get_text()
price = re.findall(r'\d+\.\d+', price_text)[0]
print(f"4. 提取价格: {price}")

# 5. 安全获取属性
first_link = soup.find('a')
data_id = first_link.get('data-id', 'N/A')
print(f"5. 安全获取属性: {data_id}")

# 6. 遍历所有子元素
container = soup.find('div', class_='container')
print(f"6. 子元素数量: {len(list(container.children))}")

## 练习题

1. 修改 MovieCrawler，添加分页功能
2. 为 TiebaCrawler 添加异常处理和重试机制
3. 实现一个通用的数据提取类，支持自定义选择器
4. 添加数据清洗功能，去除特殊字符和空白

## 注意事项

- 不同网站的 HTML 结构不同，需要根据实际情况调整选择器
- 使用浏览器开发者工具（F12）查看网页结构
- 某些网站使用 JavaScript 动态加载内容，需要使用 Selenium
- 遵守网站的爬虫协议和使用条款